In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model

model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")


agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, the supposed capital of the moon, including its weather, inhabitants, and potential social issues.\n\n## SUMMARY\nThe conversation has established that Lunapolis is the capital of the moon, with extreme weather conditions (high of 120C and low of -100C). The city is inhabited by 100,000 cheese miners, who are potentially unhappy with the new president, leading to a likely strike by the cheese miners' union.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe next steps could involve gathering more information about the cheese miners' union, the new president, and the reasons behind the potential strike, or exploring other aspects of Lunapolis and its inhabitants.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='b88f90f0-2962-4fc4-affc-6c29f4859762'),
              HumanMessage(content="If you were

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, the supposed capital of the moon, including its weather, inhabitants, and potential social issues.

## SUMMARY
The conversation has established that Lunapolis is the capital of the moon, with extreme weather conditions (high of 120C and low of -100C). The city is inhabited by 100,000 cheese miners, who are potentially unhappy with the new president, leading to a likely strike by the cheese miners' union.

## ARTIFACTS
None

## NEXT STEPS
The next steps could involve gathering more information about the cheese miners' union, the new president, and the reasons behind the potential strike, or exploring other aspects of Lunapolis and its inhabitants.


## Trim/delete messages

In [9]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [12]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [13]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='b103bde4-e32f-40ca-a372-5b94fdda671e'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='208a1c3f-e400-4ec2-a7d0-7d5fa8ad286c', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8ee1e203-7d3e-4e80-8246-71b345a73d54'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='32ecd7b9-50ad-42b7-9de4-8cea10079346', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='3ee306c8-862b-43de-9ec5-291be57baebe'),
              AIMessage(content="If the device has been exposed to extreme temperatures (very h

In [8]:
print(response["messages"][-1].content)

The device's temperature is within normal range, but I don't have the exact reading. However, I can suggest some troubleshooting steps to help you resolve the issue.

Since the device is plugged in and turned on, let's try a few things:

1. Check the power cord: Ensure the power cord is securely connected to both the device and the power outlet.
2. Check the outlet: Try plugging the device into a different outlet to rule out any issues with the electrical supply.
3. Check for loose connections: Verify that all internal components, such as the battery or RAM, are securely connected.
4. Try a hard reset: If the device has a removable battery, take it out and press the power button for 30 seconds to discharge any residual power. Then, put the battery back in and try turning it on.

If none of these steps work, it's possible that there's a hardware issue with the device. You may want to consider contacting the manufacturer or a professional repair service for further assistance.
